# noisy: DINOv3 ViT-S/16

Проверяем, помогает ли DINOv3 лучше находить шум на поверхности. Берём шесть ракурсов, сохраняем embeddings и обучаем CatBoost. Сравниваем с DINOv2 на прежней validation; internal test пока не используем.

## 1. Данные

Перед запуском в Colab включи GPU. Нужны `data/train.csv`, `data/train_png.zip` и `noisy_dinov2_assets.zip` в папке проекта на Google Drive.

In [ ]:
%pip install -q 'transformers>=4.56,<5' huggingface_hub catboost iterative-stratification tqdm

from pathlib import Path
import hashlib
import io
import json
import shutil
import zipfile

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from catboost import CatBoostClassifier
from huggingface_hub import get_token, hf_hub_download, notebook_login
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from transformers import AutoModel

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

assert torch.cuda.is_available(), 'В Colab включи GPU: Runtime → Change runtime type → T4 GPU'
device = 'cuda'
torch.manual_seed(42)
np.random.seed(42)

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data/train.csv').is_file()), None)
if ROOT is None:
    ROOT = Path('/content/drive/MyDrive/Colab Notebooks/colab_upload/AI challenge main')
assert (ROOT / 'data/train.csv').is_file(), ROOT

package = ROOT / 'noisy_dinov2_assets.zip'
if package.is_file():
    with zipfile.ZipFile(package) as archive:
        for name in archive.namelist():
            if not (ROOT / name).is_file():
                archive.extract(name, ROOT)

labels = pd.read_csv(ROOT / 'data/train.csv')
targets = labels.columns.drop('item_id').tolist()
split = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=.15, random_state=42)
train_val_idx, test_idx = next(split.split(labels, labels[targets]))
train_val = labels.iloc[train_val_idx].reset_index(drop=True)
split = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=.1765, random_state=43)
train_idx, val_idx = next(split.split(train_val, train_val[targets]))
train = train_val.iloc[train_idx].reset_index(drop=True)
val = train_val.iloc[val_idx].reset_index(drop=True)
objects = pd.concat([train, val], ignore_index=True)
assert (len(train), len(val), len(test_idx)) == (6259, 1361, 1344)
assert set(train.item_id).isdisjoint(val.item_id)
print('Train:', len(train), 'Validation:', len(val), '| noisy:', int(train.noisy.sum()), int(val.noisy.sum()))

## 2. Доступ к DINOv3

У [DINOv3 ViT-S/16](https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m) сначала нужно принять условия доступа. Если токена Hugging Face ещё нет в Colab, эта ячейка попросит его ввести.

In [ ]:
if not get_token():
    notebook_login()
else:
    print('Токен Hugging Face найден')


In [ ]:
assert get_token(), 'Сначала закончи вход в Hugging Face в предыдущей ячейке, затем запусти эту.'
MODEL_ID = 'facebook/dinov3-vits16-pretrain-lvd1689m'
hf_hub_download(repo_id=MODEL_ID, filename='config.json', token=True)
print('Доступ к DINOv3 подтверждён')

## 3. Шесть ракурсов

In [ ]:
OUT = ROOT / 'labels/noisy/models/dinov3'
OUT.mkdir(parents=True, exist_ok=True)

# ZIP с PNG копируем на диск Colab один раз, чтобы не читать Drive в каждом batch.
png_dir = ROOT / 'data/train'
if not (png_dir / f'{objects.item_id.iloc[0]}.png').is_file():
    archive_path = ROOT / 'data/train_png.zip'
    assert archive_path.is_file(), f'Нет PNG: {archive_path}'
    local_zip = Path('/content/noisy_dinov3_train_png.zip')
    if not local_zip.is_file() or local_zip.stat().st_size != archive_path.stat().st_size:
        shutil.copy2(archive_path, local_zip)
    local_dir = Path('/content/noisy_dinov3_png')
    marker = local_dir / '.ready'
    if not marker.is_file():
        local_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(local_zip) as archive:
            names = [name for name in archive.namelist() if name.lower().endswith('.png')]
            for name in tqdm(names, desc='Распаковка PNG'):
                archive.extract(name, local_dir)
        marker.touch()
    first = next(local_dir.rglob(f'{objects.item_id.iloc[0]}.png'), None)
    assert first is not None, 'Не нашёл PNG из train.csv внутри ZIP'
    png_dir = first.parent
assert (png_dir / f'{objects.item_id.iloc[-1]}.png').is_file()

transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC, antialias=True),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class Views(Dataset):
    def __len__(self):
        return len(objects)

    def __getitem__(self, row):
        item_id = objects.item_id.iloc[row]
        with Image.open(png_dir / f'{item_id}.png') as image:
            image = image.convert('RGB')
            width, height = image.size
            side = width // 3
            assert width == 3 * side and height == 2 * side, image.size
            views = [transform(image.crop((x * side, y * side, (x + 1) * side, (y + 1) * side)))
                     for y in range(2) for x in range(3)]
        return row, torch.stack(views)

print('PNG:', png_dir, '| первый объект:', Views()[0][1].shape)

## 4. Извлечение embeddings

In [ ]:
model = AutoModel.from_pretrained(MODEL_ID).to(device).eval()
model.requires_grad_(False)

# Кеш можно продолжить после прерванного запуска.
manifest = {
    'model': MODEL_ID,
    'item_ids': objects.item_id.tolist(),
    'labels_sha256': hashlib.sha256((ROOT / 'data/train.csv').read_bytes()).hexdigest(),
    'shape': [len(objects), 6, 384],
    'image_size': 224,
    'pooling': 'pooler_output',
}
cache = Path('/content/noisy_dinov3_cache')
cache.mkdir(parents=True, exist_ok=True)
files = ('embeddings.npy', 'done.npy', 'manifest.json')
if (OUT / 'manifest.json').is_file() and not (cache / 'manifest.json').is_file():
    for name in files:
        assert (OUT / name).is_file(), f'Неполный кеш: {name}'
        shutil.copy2(OUT / name, cache / name)

manifest_file = cache / 'manifest.json'
if manifest_file.is_file():
    assert json.loads(manifest_file.read_text(encoding='utf-8')) == manifest, 'Кеш от другого опыта'
    embeddings = np.load(cache / 'embeddings.npy', mmap_mode='r+')
    done = np.load(cache / 'done.npy', mmap_mode='r+')
else:
    embeddings = np.lib.format.open_memmap(cache / 'embeddings.npy', mode='w+', dtype='float32',
                                           shape=(len(objects), 6, 384))
    done = np.lib.format.open_memmap(cache / 'done.npy', mode='w+', dtype='bool', shape=(len(objects),))
    done[:] = False
    done.flush()
    manifest_file.write_text(json.dumps(manifest, ensure_ascii=False), encoding='utf-8')
assert embeddings.shape == (7620, 6, 384) and done.shape == (7620,)

def save_cache():
    embeddings.flush()
    done.flush()
    for name in files:
        temp = OUT / (name + '.tmp')
        shutil.copy2(cache / name, temp)
        temp.replace(OUT / name)

class Pending(Dataset):
    def __init__(self):
        self.rows = np.flatnonzero(~done)
        self.views = Views()

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        return self.views[int(self.rows[i])]

loader = DataLoader(Pending(), batch_size=2, shuffle=False, num_workers=0, pin_memory=True)
processed = 0
try:
    for rows, images in tqdm(loader, desc='DINOv3'):
        with torch.inference_mode():
            flat = images.flatten(0, 1).to(device)
            values = model(pixel_values=flat).pooler_output
            values = values.float().cpu().numpy().reshape(-1, 6, 384)
        assert np.isfinite(values).all()
        indexes = rows.numpy()
        embeddings[indexes] = values
        embeddings.flush()
        done[indexes] = True
        done.flush()
        processed += len(indexes)
        if processed % 250 < len(indexes):
            save_cache()
finally:
    if processed:
        save_cache()

assert done.all() and np.isfinite(embeddings).all()
print('Готово:', embeddings.shape, '| кеш:', OUT)

## 5. Геометрия и прежний результат

In [ ]:
config = json.loads((ROOT / 'archive/artifacts/catboost_features_config.json').read_text(encoding='utf-8'))
base_columns = config['feature_columns']
local_columns = ['pca_curvature_mean', 'pca_curvature_p90', 'pca_curvature_p95',
                 'normal_variation_p50', 'normal_variation_p90', 'normal_variation_p95',
                 'normal_variation_std', 'dihedral_over_10_share', 'dihedral_over_30_share']
base = pd.read_csv(ROOT / 'archive/artifacts/train_mesh_features_relative.csv').set_index('item_id')
local = pd.read_csv(ROOT / 'labels/lowpoly/artifacts/geometry_features.csv').set_index('item_id')
smooth = pd.read_csv(ROOT / 'labels/noisy/artifacts/smoothing_features.csv').set_index('item_id')
smooth_columns = [c for c in smooth if c.startswith('smooth_')]
assert len(base_columns) == 44 and len(local_columns) == 9 and len(smooth_columns) == 14
assert base.index.is_unique and local.index.is_unique and smooth.index.is_unique
geometry = base.join(local[local_columns]).join(smooth[smooth_columns])
geometry = geometry.loc[objects.item_id, base_columns + local_columns + smooth_columns].to_numpy(dtype=np.float32)
assert geometry.shape == (7620, 67) and not np.isinf(geometry).any()

previous_path = ROOT / 'labels/noisy/models/dinov2/validation_probabilities.csv'
assert previous_path.is_file(), f'Нужны вероятности предыдущего опыта: {previous_path}'
previous = pd.read_csv(previous_path).set_index('item_id').loc[val.item_id]
assert np.array_equal(previous.noisy.to_numpy(), val.noisy.to_numpy())
print('Геометрия:', geometry.shape, '| прежний DINOv2 + mesh:', round(
    f1_score(val.noisy, previous.D3_mean_max_geometry >= .35), 4))

## 6. DINOv3 + CatBoost

In [ ]:
views = np.asarray(embeddings)
mean = views.mean(axis=1)
mean_max = np.concatenate([mean, views.max(axis=1)], axis=1)
features = {
    'DINOv3 mean': mean,
    'DINOv3 mean + max': mean_max,
    'DINOv3 + geometry': np.concatenate([mean_max, geometry], axis=1),
}
y_train = train.noisy.to_numpy()
y_val = val.noisy.to_numpy()
n_train = len(train)
grid = np.arange(.05, .96, .01)

def score(p, threshold):
    pred = p >= threshold
    tn, fp, fn, tp = confusion_matrix(y_val, pred, labels=[0, 1]).ravel()
    return {'Precision': precision_score(y_val, pred, zero_division=0),
            'Recall': recall_score(y_val, pred, zero_division=0),
            'F1': f1_score(y_val, pred, zero_division=0),
            'AP': average_precision_score(y_val, p),
            'FP': int(fp), 'FN': int(fn), 'Threshold': float(threshold)}

results = [
    {'Experiment': 'DINOv2 mean + max', **score(previous.D2_mean_max.to_numpy(), .34)},
    {'Experiment': 'DINOv2 + geometry', **score(previous.D3_mean_max_geometry.to_numpy(), .35)},
]
probabilities = val[['item_id', 'noisy']].copy()
probabilities['DINOv2 mean + max'] = previous.D2_mean_max.to_numpy()
probabilities['DINOv2 + geometry'] = previous.D3_mean_max_geometry.to_numpy()

for name, X in features.items():
    candidate = CatBoostClassifier(**config['parameters'])
    candidate.fit(X[:n_train], y_train, eval_set=(X[n_train:], y_val),
                  early_stopping_rounds=50, verbose=False)
    p = candidate.predict_proba(X[n_train:])[:, 1]
    threshold = float(max(grid, key=lambda t: f1_score(y_val, p >= t)))
    results.append({'Experiment': name, **score(p, threshold)})
    probabilities[name] = p
    candidate.save_model(str(OUT / (name.replace(' ', '_').replace('+', 'and') + '.cbm')))
    print(name, '| F1:', round(results[-1]['F1'], 4), '| AP:', round(results[-1]['AP'], 4), flush=True)

table = pd.DataFrame(results).sort_values('F1', ascending=False)
display(table.round(4))
table.to_csv(OUT / 'metrics.csv', index=False)
probabilities.to_csv(OUT / 'validation_probabilities.csv', index=False)

## 7. Где ошибается DINOv3 + геометрия

In [ ]:
best = table[table.Experiment.eq('DINOv3 + geometry')].iloc[0]
review = val[['item_id', 'noisy', 'lowpoly', 'artifacts']].copy()
review['probability'] = probabilities[best.Experiment].to_numpy()
review['prediction'] = (review.probability >= best.Threshold).astype(int)
for label in ('lowpoly', 'artifacts'):
    group = review[(review.noisy == 1) & (review[label] == 1)]
    print(f'noisy + {label}: {len(group)} объектов, {int((group.prediction == 0).sum())} пропущено')

old_pred = previous.D3_mean_max_geometry.to_numpy() >= .35
new_pred = probabilities['DINOv3 + geometry'].to_numpy() >= table.loc[
    table.Experiment == 'DINOv3 + geometry', 'Threshold'].iloc[0]
print('DINOv3 исправила FN DINOv2:', int(((y_val == 1) & ~old_pred & new_pred).sum()),
      '| новые FN:', int(((y_val == 1) & old_pred & ~new_pred).sum()))
print('DINOv3 исправила FP DINOv2:', int(((y_val == 0) & old_pred & ~new_pred).sum()),
      '| новые FP:', int(((y_val == 0) & ~old_pred & new_pred).sum()))

for title, frame in [('FP', review.query('noisy == 0 and prediction == 1')),
                     ('FN', review.query('noisy == 1 and prediction == 0'))]:
    sample = frame.sort_values('probability', ascending=title == 'FN').head(2)
    display(sample)
    for item_id in sample.item_id:
        with Image.open(png_dir / f'{item_id}.png') as image:
            plt.figure(figsize=(12, 6))
            plt.imshow(image.convert('RGB'))
            plt.title(f'{title}: {item_id}', color='#0098F8')
            plt.axis('off')
            plt.show()

## Вывод

Сравниваем F1 DINOv3 + геометрия с 0.8301 у DINOv2 + геометрия. Все пороги и варианты моделей выбраны на одной validation, поэтому результат может быть оптимистичным. Internal test не использовали.